In [1]:
import warnings
warnings.filterwarnings("ignore")
import good
import json
import os
import pandas as pd
import importlib
from good import s_runner
from good import helper
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from pathlib import Path

In [2]:
# =============================================================================
# Input settings
# =============================================================================

import os
import json
import importlib
import pandas as pd

# ---------------------------------------------------------
# Model region
# ---------------------------------------------------------
# This is the model region key, not one IPM node.
# For PJM, this maps to several GOOD nodes.
N_SCENARIO_WORKERS = 5
CPLEX_THREADS_PER_SCENARIO = 10
IPM_REGION = "SERC-N"
MODEL_REGION = IPM_REGION
STATE = IPM_REGION

YEAR_INPUT = 2030
MONTH_INPUT = 0
DAY_INPUT = 360

Discount_rate = 0.07
Lifetime = 25

# ---------------------------------------------------------
# Load EV data and scenarios
# ---------------------------------------------------------
# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
def find_good_root():
    """
    Find the GOOD project root in a portable way.

    Priority:
    1. Use GOOD_ROOT environment variable if it exists.
    2. Search upward from the current working directory.
    """
    env_root = os.environ.get("GOOD_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / "good").exists() and (folder / "Examples").exists():
            return folder

    raise FileNotFoundError(
        "Could not find GOOD project root. "
        "Set GOOD_ROOT to the path of your GOOD folder."
    )


GOOD_ROOT = find_good_root()
EV_DATA_DIR = GOOD_ROOT / "Examples" / "EVDATA"

ev_data_path = EV_DATA_DIR / f"{IPM_REGION}_EVDATA.json"

if not ev_data_path.exists():
    available_files = sorted(EV_DATA_DIR.glob("*.json"))
    available_names = [p.name for p in available_files]

    raise FileNotFoundError(
        f"Could not find EV data file:\n"
        f"  {ev_data_path}\n\n"
        f"Available files in {EV_DATA_DIR}:\n"
        f"  {available_names}"
    )

with ev_data_path.open("r") as f:
    ev_data = json.load(f)

SCENARIOS = helper.load_scenarios(IPM_REGION, YEAR_INPUT)

# ---------------------------------------------------------
# Load base graph and base policies
# ---------------------------------------------------------
BASE_GRAPH = good.graph.graph_from_json(f"Examples/Nodes/{IPM_REGION}_IPM.json")
BASE_POLICIES = good.utilities.read_json("Examples/policies.json")

# ---------------------------------------------------------
# Run settings
# ---------------------------------------------------------
TARGET_PEAK_GW = 150
ADOPTION_SCENARIOS = [
    "slow",
    "mid",
    "fast"
]
CHARGING_SCENARIOS = {
    "midnight": {
        "profile": "timed_charging",
        "description": "Midnight timed charging",
    },
    "delay": {
        "profile": "max_delay",
        "description": "Maximum delay charging",
    },
    "arrive": {
        "profile": "min_delay",
        "description": "Immediate arrival charging",
    },
    "flex": {
        "profile": "load_leveling",
        "description": "Flexible load leveling",
    },
}
# ---------------------------------------------------------
# GOOD regions used by this model region
# ---------------------------------------------------------

SERC_N_REGIONS = [
    "S_C_KY",
    "S_D_AECI",
    "S_C_TVA",
]

STATE_TO_REGIONS = {
    "SERC-N": SERC_N_REGIONS,
}

# ---------------------------------------------------------
# Battery storage distribution weights
# ---------------------------------------------------------
# EIA-860M operating battery capacity, June 2026:
#
# S_C_KY       0.0 MW
# S_D_AECI     0.0 MW
# S_C_TVA    152.5 MW
#
# Total      152.5 MW
#
# The S_C_TVA operating battery capacity includes:
#
# Golden Triangle                    50.0 MW
# Golden Triangle II                 50.0 MW
# Optimist                           50.0 MW
# Walnut Grove Demonstration Plant    1.5 MW
# Redstone Arsenal Hybrid             1.0 MW
#
# Pumped-storage hydro is excluded because these weights
# are specifically for battery storage.
#
# Planned batteries that are not operating as of June 2026
# are also excluded.

BATTERY_WEIGHTS = {
    "S_C_KY": 0.0000,
    "S_D_AECI": 0.0000,
    "S_C_TVA": 1.0000,
}

assert abs(sum(BATTERY_WEIGHTS.values()) - 1.0) < 1e-6


# =============================================================================
# Policies used by run_one_scenario
# =============================================================================

# ---------------------------------------------------------
# Region-level retirement policies
# ---------------------------------------------------------
# The denominator is the EPA NEEDS Winter 2024 active fleet.
#
# The retirement assumptions are updated using June 2026
# information from EIA and the affected utilities.
#
# Units already placed in the NEEDS "retire by 2028" sheet
# are not part of the active fleet and are not retired again.
#
# This is particularly important for:
#
# Mill Creek Units 1-2
# Cumberland Unit 2
# Kingston Units 1-9
# Older Johnsonville combustion turbines
#
# These units are already absent from the active fleet used
# by the model.

RETIREMENT_POLICIES = {
    "SERC-N": {
        # No confirmed coal retirement is applied to the
        # model's active fleet through 2030.
        #
        # EPA NEEDS originally assigned a 2029 retirement to:
        #
        # E.W. Brown Unit 3: 412 MW
        # Cumberland Unit 1: 1,239 MW
        #
        # However, these assumptions are now outdated:
        #
        # 1. LG&E/KU's updated planning places E.W. Brown
        #    Unit 3 retirement in 2035.
        #
        # 2. In February 2026, the TVA Board authorized steps
        #    to continue operating Cumberland and Kingston
        #    beyond their previously scheduled retirement dates.
        #
        # Therefore, neither retirement is imposed in the
        # 2030 central scenario.
        "coal": 0.0000,

        # No matched O/G steam retirement through 2030.
        #
        # Active O/G steam capacity: 5.5 MW
        "oil": 0.0000,

        # Macon Unit 4:
        # 0.7 MW planned retirement in EIA-860M.
        #
        # 0.7 / 6,676.2 MW active combustion-turbine capacity
        "natural gas turbine": 0.0001,

        # No matched retirement through 2030.
        #
        # Active combined-cycle capacity: 12,445.0 MW
        "natural gas combined cycle": 0.0000,

        # No matched retirement through 2030.
        #
        # Active nuclear capacity: 8,184.4 MW
        "nuclear": 0.0000,
    }
}


# ---------------------------------------------------------
# Asset constraint policies
# ---------------------------------------------------------

ASSET_CONSTRAINT_POLICIES = {
    "SERC-N": {
        # SERC-N 2030 central logic:
        #
        # The combined active thermal fleet contains approximately:
        #
        # Nuclear:              8,184.4 MW
        # Coal steam:          10,584.0 MW
        # Combined cycle:      12,445.0 MW
        # Combustion turbine:   6,676.2 MW
        # O/G steam:                5.5 MW
        #
        # The region also contains substantial TVA hydro generation
        # and approximately 1,585.8 MW of wind in S_D_AECI.
        #
        # Nuclear remains highly must-run because TVA's nuclear fleet
        # provides a large share of stable regional generation.
        #
        # Coal receives a 20% aggregated minimum-output floor. This
        # represents continued operation of part of the coal fleet
        # following TVA's 2026 decision to pursue extended operation.
        # The floor remains low enough to allow wind, solar, hydro,
        # and lower-load conditions.
        #
        # Combined-cycle gas is the largest thermal category and
        # provides both energy and reliability. A 15% floor represents
        # the portion likely to remain committed without forcing the
        # entire combined-cycle fleet to behave as baseload.
        #
        # Combustion turbines are peaking resources and receive no
        # must-run floor.
        #
        # O/G steam capacity is extremely small in this model region.
        # The standard 10% floor and 8% hourly transition limit are
        # retained for consistency, but they have almost no effect
        # because only 5.5 MW is present.

        "nuclear": {
            "must_run_fraction": 0.95,
            "ramp_rate": 0.03,
        },

        "coal": {
            "must_run_fraction": 0.20,
            "ramp_rate": 0.05,
        },

        "natural gas combined cycle": {
            "must_run_fraction": 0.15,
            "ramp_rate": 0.20,
        },

        "natural gas turbine": {
            "must_run_fraction": 0.00,
            "ramp_rate": 0.80,
        },

        "oil": {
            "must_run_fraction": 0.10,
            "ramp_rate": 0.08,
        },
    },
}

FLEXIBILITY_POLICIES = {
    "default": {
        # -----------------------------
        # V1G settings
        # -----------------------------
        "v1g": {
            "base_shift_cost": 0.0,
            "fixed_om_per_kw_year": 0.0,
            "shift_window_hours": 24,
        },

        # -----------------------------
        # V2G settings
        # -----------------------------
        "v2g": {
            "window_hours": 24,
            "energy_duration_hours": 1,
            "roundtrip_efficiency": 0.985,
            "base_shift_cost": 1.3784e-9,
            "fixed_om_per_kw_year": 0.0,
        },

        # -----------------------------
        # Stationary battery settings
        # -----------------------------
        "battery": {
            "duration_hours": 4,
            "charge_efficiency": 0.93,
            "discharge_efficiency": 0.92,
            "fixed_om_per_kw_year": 3.75,
            "cycling_cost_per_mwh": 0.01,
            "initial_soc_fraction": 0.50,
            "total_power_mw": 152,
        },
    },
    "SERC-N": {
        "battery": {
            "total_power_mw":152,
        },
    },
}

# ---------------------------------------------------------
# Economic policies
# ---------------------------------------------------------
ECONOMIC_POLICIES = {
    "default": {
        "discount_rate": 0.07,
        "lifetime_years": 25,
        "import_operating_cost": 1.75e-8,
        "apply_crf_to_renewables": True,
        "apply_crf_to_storage": True,
        "renewable_fuels_for_crf": {"solar", "wind"},
    },

    "SERC-N": {},
}

# ---------------------------------------------------------
# Transmission policies
# ---------------------------------------------------------
TRANSMISSION_POLICIES = {
    "default": {
        "apply_distance_enhancement": True,
        "default_operating_cost": 2.222222222222222e-09,
        "default_efficiency": 0.90,
        "overwrite_existing_operating_cost": False,
        "overwrite_existing_efficiency": False,
    },
    "SERC-N": {},
}

# ---------------------------------------------------------
# Policy inputs passed directly to run_one_scenario
# ---------------------------------------------------------
# Important:
# For PJM, state_rps_policies must be None.
# Each PJM scenario already has cfg["state_rps_policies"].
# This is how rps_minus10, rps_base, and rps_plus10 work.
POLICY_INPUTS = {
    "state_rps_policies": None,
}

In [3]:
"""
Professional Scenario Runner - Loops Through All Adoption × Charging Scenarios
Runs all combinations of adoption levels and charging patterns separately.
Each combination gets its own folder.
"""
# =============================================================================
# Main Loop - Process Each Adoption × Charging Scenario Combination
# =============================================================================

importlib.reload(s_runner)


total_combinations = len(ADOPTION_SCENARIOS) * len(CHARGING_SCENARIOS)
total_runs = len(SCENARIOS) * total_combinations
print(f"\n{'#'*80}")
print(f"# RUNNING ALL ADOPTION × CHARGING SCENARIOS")
print(f"# Adoption scenarios: {len(ADOPTION_SCENARIOS)} ({', '.join(ADOPTION_SCENARIOS)})")
print(f"# Charging scenarios: {len(CHARGING_SCENARIOS)} ({', '.join(CHARGING_SCENARIOS.keys())})")
print(f"# Total combinations: {total_combinations}")
print(f"# Total runs: {total_runs} (= {len(SCENARIOS)} scenarios × {total_combinations} combinations)")
print(f"{'#'*80}\n")
# Track overall progress
all_results_tracker = {}

for adoption_level in ADOPTION_SCENARIOS:

    print(f"\n{'█'*80}")
    print(f"█ ADOPTION LEVEL: {adoption_level.upper()}")
    print(f"{'█'*80}\n")

    for charging_name, charging_config in CHARGING_SCENARIOS.items():

        # Set up this combination
        RESULTS_DIR = f"Output/{IPM_REGION}/scenario_results_{YEAR_INPUT}_{adoption_level}_{IPM_REGION}_{charging_name}"
        CHARGING_PROFILE = charging_config["profile"]


        ctx = s_runner.ScenarioRunContext(
            scenarios=SCENARIOS,
            base_graph=BASE_GRAPH,
            base_policies=BASE_POLICIES,
            state_to_regions=STATE_TO_REGIONS,
            ev_data=ev_data,
            battery_weights=BATTERY_WEIGHTS,
            results_dir=RESULTS_DIR,
            discount_rate=Discount_rate,
            lifetime=Lifetime,
            retirement_policies=RETIREMENT_POLICIES,
            asset_constraint_policies=ASSET_CONSTRAINT_POLICIES,
            solver_threads=CPLEX_THREADS_PER_SCENARIO,
        )
        # Create unique key for tracking
        combo_key = f"{adoption_level}_{charging_name}"

        print(f"\n{'='*80}")
        print(f"COMBINATION: {adoption_level.upper()} adoption + {charging_config['description']}")
        print(f"Results Directory: {RESULTS_DIR}")
        print(f"{'='*80}\n")

        # Create results directory
        os.makedirs(RESULTS_DIR, exist_ok=True)

        # Run all scenarios for this combination
        all_results = []
        failed_scenarios = []

        tasks = []

        for scenario_id in sorted(SCENARIOS.keys()):
            tasks.append({
                "scenario_id": scenario_id,
                "ctx": ctx,
                "year": YEAR_INPUT,
                "adoption": adoption_level,
                "charging": CHARGING_PROFILE,
                "charging_name": charging_name,
                "charging_description": charging_config["description"],
                "month": MONTH_INPUT,
                "day_duration": DAY_INPUT,
                "model_region": IPM_REGION,
                "discount_rate": Discount_rate,
                "lifetime": Lifetime,
                "fix_peak": False,
                "target_peak_gw": TARGET_PEAK_GW,
                "peak_region_mode": "all",
                "peak_regions": None,
                "retirement_policy": None,
                "asset_constraint_policy": None,
                "use_state_default_retirement": True,
                "use_state_default_asset_constraints": True,
                "flexibility_policy": FLEXIBILITY_POLICIES,
                "use_state_default_flexibility": True,
                "economic_policy": ECONOMIC_POLICIES,
                "use_state_default_economic_policy": True,
                "transmission_policy": TRANSMISSION_POLICIES,
                "use_state_default_transmission_policy": True,
            })
       

        mp_context = mp.get_context("spawn")

        with ProcessPoolExecutor(
            max_workers=N_SCENARIO_WORKERS,
            mp_context=mp_context,
        ) as executor:

            futures = {
                executor.submit(s_runner.run_one_scenario_parallel_task, task): task["scenario_id"]
                for task in tasks
            }

            for future in as_completed(futures):
                scenario_id = futures[future]
                result = future.result()

                if result["ok"]:
                    all_results.append(result["row"])
                    print(f"  Scenario {scenario_id:>3} ✓")
                else:
                    print(f"  Scenario {scenario_id:>3} ✗ Error: {result['error']}")

                    failed_scenarios.append({
                        "scenario_id": scenario_id,
                        "error": result["error"],
                        "traceback": result["traceback"],
                    })
        
        
        # Save results for this combination
        all_results_df = pd.DataFrame(all_results)
        summary_path = os.path.join(RESULTS_DIR, "all_scenarios_summary.csv")
        all_results_df.to_csv(summary_path, index=False)

        # Save failed scenarios if any
        if failed_scenarios:
            failed_df = pd.DataFrame(failed_scenarios)
            failed_path = os.path.join(RESULTS_DIR, "failed_scenarios.csv")
            failed_df.to_csv(failed_path, index=False)

        # Store for later comparison
        all_results_tracker[combo_key] = all_results_df

        # Print summary for this combination
        print(f"\n  Summary:")
        print(f"    Successful: {len(all_results)}/{len(SCENARIOS)}")
        print(f"    Failed: {len(failed_scenarios)}/{len(SCENARIOS)}")
        print(f"    Saved to: {summary_path}")

        if failed_scenarios:
            print(f"    ⚠ Failed scenarios logged to: {failed_path}")


################################################################################
# RUNNING ALL ADOPTION × CHARGING SCENARIOS
# Adoption scenarios: 3 (slow, mid, fast)
# Charging scenarios: 4 (midnight, delay, arrive, flex)
# Total combinations: 12
# Total runs: 180 (= 15 scenarios × 12 combinations)
################################################################################


████████████████████████████████████████████████████████████████████████████████
█ ADOPTION LEVEL: SLOW
████████████████████████████████████████████████████████████████████████████████


COMBINATION: SLOW adoption + Midnight timed charging
Results Directory: Output/SERC-N/scenario_results_2030_slow_SERC-N_midnight

EV state: SERC-N
Model region: SERC-N
STATE_TO_REGIONS key used: SERC-N
GOOD regions used: ['S_C_KY', 'S_D_AECI', 'S_C_TVA']
Base load peak scaling is OFF. Using original graph load.
Created RPS policies:
  rps_SERC-N_MO: ratio=0.05, regions=['S_C_KY', 'S_D_AECI', 'S_C_TVA']
  rps_SERC-N_NC: ratio